# AthleteLoad: 01. Exploratory Data Analysis

This notebook explores longitudinal athlete monitoring data from the SoccerMon dataset. We analyze subjective training load, daily wellness questionnaires, and injury report distributions across two elite women's football teams over two seasons (2020 and 2021).

In [ ]:
import os
import sys
# Ensure parent directory is in python path
if os.path.abspath('..') not in sys.path:
    sys.path.insert(0, os.path.abspath('..'))

def get_path(rel_path):
    if os.path.exists(rel_path):
        return rel_path
    elif os.path.exists(os.path.join('..', rel_path)):
        return os.path.join('..', rel_path)
    return rel_path

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import setup_plotting_style, PALETTE
setup_plotting_style()

df = pd.read_csv(get_path('data/processed/player_daily_features.csv'))
df['date'] = pd.to_datetime(df['date'])
print(f'Dataset Shape: {df.shape[0]:,} player-days across {df["player_id"].nunique()} athletes.')
print(f'Date Range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Total Reported Injury Days: {df["injury_reported"].sum()}')
print(f'Total 7-Day Pre-Injury Windows: {df["injury_next_7_days"].sum()} ({df["injury_next_7_days"].mean():.2%})')

## 1. Reported Injuries by Anatomical Location & Severity

In [ ]:
inj_raw = pd.read_csv(get_path('data/raw/subjective/injury/injury.csv'))
locations_list = []
for s in inj_raw['type']:
    try:
        d = json.loads(s)
        for loc, sev in d.items():
            locations_list.append({'location': loc.replace('_', ' ').title(), 'severity': sev.capitalize()})
    except Exception:
        pass
inj_loc_df = pd.DataFrame(locations_list)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
loc_counts = inj_loc_df['location'].value_counts().head(8)
sns.barplot(x=loc_counts.values, y=loc_counts.index, ax=axes[0], palette='Blues_r', hue=loc_counts.index, legend=False)
axes[0].set_title('Reported Injuries by Anatomical Location', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Number of Reports', fontweight='bold')
axes[0].set_ylabel('Anatomical Location', fontweight='bold')

sev_df = inj_loc_df.groupby(['location', 'severity']).size().unstack(fill_value=0)
sev_df['total'] = sev_df.sum(axis=1)
sev_top = sev_df.sort_values('total', ascending=False).head(8).drop(columns=['total'])
sev_top.plot(kind='barh', stacked=True, ax=axes[1], color=[PALETTE['healthy'], PALETTE['danger']], edgecolor='black', linewidth=0.5)
axes[1].set_title('Injury Severity Proportion by Location', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Number of Reports', fontweight='bold')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

## 2. Longitudinal Athlete Training Load Trajectories

In [ ]:
sample_pid = 'TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93'
p_df = df[df['player_id'] == sample_pid].sort_values('date').copy()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True, gridspec_kw={'height_ratios': [2.2, 1.2]})
ax1.bar(p_df['date'], p_df['daily_load'], color='#bdc3c7', alpha=0.5, label='Daily Load (sRPE)', width=1.0)
ax1.plot(p_df['date'], p_df['atl_7d'], color=PALETTE['primary'], linewidth=2.0, label='ATL (7-day Acute Load)')
ax1.plot(p_df['date'], p_df['ctl_28d'] * 7, color=PALETTE['accent'], linewidth=2.0, label='CTL (28-day Chronic Load x 7)')

inj_dates = p_df[p_df['injury_reported'] == 1]['date']
for idt in inj_dates:
    ax1.axvline(idt, color=PALETTE['danger'], linestyle='--', linewidth=1.5, alpha=0.8)
if len(inj_dates) > 0:
    ax1.plot([], [], color=PALETTE['danger'], linestyle='--', label='Reported Injury Event')

ax1.set_title(f'Longitudinal Workload Trajectory and Injury Events (Athlete: {sample_pid[:12]})', fontsize=12, fontweight='bold')
ax1.set_ylabel('Workload (sRPE Units)', fontweight='bold')
ax1.legend(loc='upper right', frameon=True)

ax2.plot(p_df['date'], p_df['acwr_calc'], color=PALETTE['secondary'], linewidth=1.8, label='Calculated ACWR (7:28)')
ax2.axhspan(0.8, 1.3, color='#2ecc71', alpha=0.2, label="Optimal 'Sweet Spot' (0.8 - 1.3)")
ax2.axhline(1.5, color=PALETTE['danger'], linestyle=':', linewidth=1.5, label='High Risk Threshold (> 1.5)')
ax2.set_ylim(0, 3.0)
ax2.set_ylabel('ACWR', fontweight='bold')
ax2.set_xlabel('Date', fontweight='bold')
ax2.legend(loc='upper right', frameon=True)
plt.tight_layout()
plt.show()

## 3. Wellness & Workload Correlation Heatmap

In [ ]:
corr_cols = ['fatigue', 'soreness', 'mood', 'readiness', 'sleep_duration', 'sleep_quality', 'stress', 'atl', 'acwr']
corr_labels = ['Fatigue', 'Soreness', 'Mood', 'Readiness', 'Sleep Dur', 'Sleep Qual', 'Stress', 'ATL', 'ACWR']
corr_mat = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(
    corr_mat, mask=mask, annot=True, fmt='.2f', cmap='vlag', vmin=-1, vmax=1,
    square=True, linewidths=0.5, cbar_kws={'shrink': 0.8, 'label': 'Pearson Correlation'},
    xticklabels=corr_labels, yticklabels=corr_labels, ax=ax
)
ax.set_title('Correlation Matrix of Athlete Wellness and Workload Metrics', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Pre-Injury vs Baseline Workload & Wellness Comparisons

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
metrics_to_plot = [
    ('atl', 'Acute Training Load (ATL)', 'sRPE units'),
    ('acwr', 'Acute:Chronic Ratio (ACWR)', 'Ratio'),
    ('load_7d_sum', '7-Day Load Sum', 'sRPE units')
]
for i, (col, title, unit) in enumerate(metrics_to_plot):
    valid = df[[col, 'injury_next_7_days']].dropna().copy()
    valid['Status'] = valid['injury_next_7_days'].map({0: 'Healthy Baseline', 1: 'Pre-Injury (7d)'})
    sns.boxplot(x='Status', y=col, data=valid, ax=axes[i], palette=[PALETTE['healthy'], PALETTE['danger']], showfliers=False, width=0.45, hue='Status', legend=False)
    axes[i].set_title(title, fontweight='bold', fontsize=11)
    axes[i].set_ylabel(unit, fontweight='bold')
    axes[i].set_xlabel('')
plt.tight_layout()
plt.show()